# 11-2. 종일 음이항 NB2 모형 튜닝

2025년 종일 서울행 광역버스 승차인원을 종속변수로 하고 `log(2024년 승차인원)`을 offset으로 적용한다. 분석 단위는 셀×노선이며 노선 고정효과와 셀 단위 군집 강건 표준오차를 사용한다. 모든 연속형 변수는 표준화하지 않은 원단위이다.

- 기본모형: 보행 더미 2개 + IC + 경쟁 목적지 비율 + 2024년 20~50대 인구수 + 건축물 수
- 튜닝 1: 보행 20분 이내 더미만 사용하고 나머지는 기본모형과 동일
- 튜닝 2: 기본모형의 2024년 인구수를 2024→2025년 20~50대 인구 증감으로 교체


In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial
from IPython.display import display

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
import config as cfg
cfg = importlib.reload(cfg)
cfg.setup_notebook()
ID, ROUTE = '셀 ID', '노선'
print('설정 파일:', Path(cfg.__file__).resolve())
print('경쟁목적지 파일:', cfg.COMPETING_CELL_ROUTE_PATH)


## 1. 공통 데이터 구축

In [ ]:
grid = cfg.load_model_data()
routes = cfg.load_cell_route_all_day()
grid_cols = [ID, 'walk_20', 'walk_20_60', 'ic_access', 'pop_20_50', 'buildings']
model_data = routes.merge(grid[grid_cols], on=ID, how='inner', validate='many_to_one')

pop = cfg.read_csv(cfg.POPULATION_CHANGE_PATH)
pop[ID] = pd.to_numeric(pop[ID], errors='coerce').astype('Int64')
pop_col = next(c for c in pop.columns if '증감(100명)' in str(c))
pop['pop_change'] = pd.to_numeric(pop[pop_col], errors='coerce') * 100
model_data = model_data.merge(pop[[ID, 'pop_change']], on=ID, how='left', validate='many_to_one')

numeric_cols = ['y2024','y2025','walk_20','walk_20_60','ic_access','competing_share','pop_20_50','pop_change','buildings']
for c in numeric_cols:
    model_data[c] = pd.to_numeric(model_data[c], errors='coerce')
model_data = model_data[model_data['y2024'] > 0].dropna(subset=numeric_cols).reset_index(drop=True)
print('관측치:',len(model_data),'| 격자:',model_data[ID].nunique(),'| 노선:',model_data[ROUTE].nunique())
print('셀×노선 중복:',model_data.duplicated([ID,ROUTE]).sum())
print('경쟁 목적지 비율 결측:',model_data['competing_share'].isna().sum())
display(model_data[[ID,ROUTE]+numeric_cols].head())


## 2. NB2 추정 함수

In [ ]:
LABELS={'walk_20':'GTX 보행 20분 이내','walk_20_60':'GTX 보행 20~60분','ic_access':'IC까지 도로거리(km)','competing_share':'GTX 경쟁 목적지 비율(%p)','pop_20_50':'2024년 20~50대 인구(명)','pop_change':'2024→2025년 20~50대 인구 증감(명)','buildings':'2024년 건축물 수(동)'}

def fit_nb2(name, features):
    d=model_data.dropna(subset=['y2024','y2025']+features).copy()
    x=d[features].astype(float)  # 원단위, 표준화 없음
    route_fe=pd.get_dummies(d[ROUTE].astype(str),prefix='노선',drop_first=True,dtype=float)
    X=sm.add_constant(pd.concat([x,route_fe],axis=1),has_constant='add')
    offset=np.log(d['y2024'].astype(float))
    poisson=sm.GLM(d['y2025'],X,family=sm.families.Poisson(),offset=offset).fit()
    model=NegativeBinomial(d['y2025'],X,offset=offset,loglike_method='nb2')
    fit=model.fit(start_params=np.r_[np.asarray(poisson.params),0.2],method='bfgs',maxiter=5000,disp=False,cov_type='cluster',cov_kwds={'groups':pd.factorize(d[ID])[0],'use_correction':True})
    if not fit.mle_retvals.get('converged') or fit.mle_retvals.get('warnflag',0)!=0:
        raise RuntimeError(f'{name} 수렴 실패: {fit.mle_retvals}')
    null=NegativeBinomial(d['y2025'],np.ones((len(d),1)),offset=offset,loglike_method='nb2').fit(start_params=np.array([np.log(d['y2025'].sum()/d['y2024'].sum()),0.2]),method='bfgs',maxiter=5000,disp=False)
    ci=fit.conf_int().loc[features]
    result=pd.DataFrame({'변수':[LABELS[v] for v in features],'계수':[fit.params[v] for v in features],'IRR':[np.exp(fit.params[v]) for v in features],'상대적 차이(%)':[(np.exp(fit.params[v])-1)*100 for v in features],'z값':[fit.tvalues[v] for v in features],'p-value':[fit.pvalues[v] for v in features],'95% CI 하한(%)':[(np.exp(ci.loc[v,0])-1)*100 for v in features],'95% CI 상한(%)':[(np.exp(ci.loc[v,1])-1)*100 for v in features]})
    info={'모형':name,'표본 수':len(d),'격자 수':d[ID].nunique(),'노선 수':d[ROUTE].nunique(),'로그우도':fit.llf,'McFadden pseudo R²':1-fit.llf/null.llf,'AIC':fit.aic,'BIC':fit.bic,'alpha':fit.params['alpha'],'수렴':fit.mle_retvals.get('converged')}
    return {'fit':fit,'data':d,'X':X,'offset':offset,'result':result,'info':info}


## 3. 기본모형 — 보행 더미 2개와 2024년 인구수

In [ ]:
BASE_FEATURES=['walk_20','walk_20_60','ic_access','competing_share','pop_20_50','buildings']
base_model=fit_nb2('기본모형',BASE_FEATURES)
display(pd.DataFrame([base_model['info']]).round(4))
display(base_model['result'].round(4))


## 4. 튜닝 1 — 보행 20분 이내 더미만 사용

In [ ]:
TUNE1_FEATURES=['walk_20','ic_access','competing_share','pop_20_50','buildings']
tune1_model=fit_nb2('튜닝1_20분더미만',TUNE1_FEATURES)
display(pd.DataFrame([tune1_model['info']]).round(4))
display(tune1_model['result'].round(4))


## 5. 튜닝 2 — 기존 인구수를 2024→2025년 인구 증감으로 교체

In [ ]:
TUNE2_FEATURES=['walk_20','walk_20_60','ic_access','competing_share','pop_change','buildings']
tune2_model=fit_nb2('튜닝2_인구증감',TUNE2_FEATURES)
display(pd.DataFrame([tune2_model['info']]).round(4))
display(tune2_model['result'].round(4))


## 6. 세 모형 비교

In [ ]:
models=[base_model,tune1_model,tune2_model]
comparison=pd.DataFrame([m['info'] for m in models])
display(comparison.round(4))
rows=[]
for m in models:
    t=m['result'][['변수','계수','IRR','상대적 차이(%)','p-value']].copy()
    t.insert(0,'모형',m['info']['모형'])
    rows.append(t)
display(pd.concat(rows,ignore_index=True).round(4))
print('AIC·BIC는 같은 표본의 모형끼리 낮을수록 좋습니다.')
print('변수 채택은 적합도뿐 아니라 연구 가설, 계수 방향, 신뢰구간과 p-value를 함께 판단합니다.')
